# GPU TCN training — electricity demand forecasting

This notebook trains a **Temporal Convolutional Network (TCN)** using the arrays created by `TCN_data_preparation.ipynb`. It is deliberately GPU-enforced: it stops before training if no TensorFlow GPU is available, and disables soft placement so model operations cannot silently fall back to CPU.

Data loading and saving necessarily use the host machine, but the model forward pass, gradient calculation, and optimizer updates are required to run on the GPU.

In [1]:
# Run this cell first, before constructing any TensorFlow tensors/models.
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "1"

from pathlib import Path
import json
import platform
import sys
import joblib
import numpy as np
import pandas as pd
import sklearn
import tensorflow as tf

DATA_DIR = (Path.cwd().parent / "data")
PREPARED_DIR = DATA_DIR / "tcn_prepared"
TRAINING_DIR = DATA_DIR / "tcn_training"
TRAINING_DIR.mkdir(parents=True, exist_ok=True)

CITY = "las_vegas"       # change to "phoenix" to train that city's independent model
BATCH_SIZE = 64
EPOCHS = 100
LEARNING_RATE = 1e-3
USE_MIXED_PRECISION = True  # recommended for recent NVIDIA GPUs
SEED = 42

tf.keras.utils.set_random_seed(SEED)
np.random.seed(SEED)
assert PREPARED_DIR.exists(), (
    f"Prepared data not found at {PREPARED_DIR}. Run TCN_data_preparation.ipynb first."
)

## Required GPU sanity check

This cell prints all relevant library versions, identifies the GPU, verifies a real TensorFlow operation is placed on `/GPU:0`, and raises an error if the GPU is unavailable. Do not bypass it.

In [2]:
# GPU and version sanity check — training is blocked unless this passes.
print("Python:     ", sys.version.split()[0])
print("Platform:   ", platform.platform())
print("NumPy:      ", np.__version__)
print("pandas:     ", pd.__version__)
print("scikit-learn:", sklearn.__version__)
print("joblib:     ", joblib.__version__)
print("TensorFlow: ", tf.__version__)
print("Keras:      ", tf.keras.__version__)
print("Built with CUDA:", tf.test.is_built_with_cuda())

physical_gpus = tf.config.list_physical_devices("GPU")
if not physical_gpus:
    raise RuntimeError(
        "NO GPU DETECTED. Training has been stopped. Install a CUDA-enabled TensorFlow build and compatible NVIDIA driver, then restart the kernel."
    )

# Must be configured before a model or tensor is created.
for gpu in physical_gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as exc:
        print(f"Memory-growth configuration note: {exc}")

# No implicit CPU fallback for TensorFlow model operations.
tf.config.set_soft_device_placement(False)
logical_gpus = tf.config.list_logical_devices("GPU")
print(f"Physical GPUs: {physical_gpus}")
print(f"Logical GPUs:  {logical_gpus}")
for gpu in physical_gpus:
    print("GPU details:   ", tf.config.experimental.get_device_details(gpu))

# Proves that a real TensorFlow kernel executes on GPU:0.
with tf.device("/GPU:0"):
    a = tf.random.normal((1024, 1024), seed=SEED)
    gpu_probe = tf.linalg.matmul(a, a)
assert "GPU" in gpu_probe.device.upper(), f"GPU probe ran on {gpu_probe.device}, not a GPU"
print(f"GPU execution verified: {gpu_probe.device}")
del a, gpu_probe

if USE_MIXED_PRECISION:
    tf.keras.mixed_precision.set_global_policy("mixed_float16")
print("Mixed precision policy:", tf.keras.mixed_precision.global_policy())
print("SANITY CHECK PASSED — GPU-only TCN training is enabled.")

Python:      3.10.20
Platform:    Windows-10-10.0.26200-SP0
NumPy:       1.26.4
pandas:      2.3.3
scikit-learn: 1.7.2
joblib:      1.5.3
TensorFlow:  2.10.0
Keras:       2.10.0
Built with CUDA: True
Physical GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Logical GPUs:  [LogicalDevice(name='/device:GPU:0', device_type='GPU')]
GPU details:    {'device_name': 'NVIDIA GeForce RTX 3050 6GB Laptop GPU', 'compute_capability': (8, 6)}
GPU execution verified: /job:localhost/replica:0/task:0/device:GPU:0
INFO:tensorflow:Mixed precision compatibility check (mixed_float16): OK
Your GPU will likely run quickly with dtype policy mixed_float16 as it has compute capability of at least 7.0. Your GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU, compute capability 8.6
Mixed precision policy: <Policy "mixed_float16">
SANITY CHECK PASSED — GPU-only TCN training is enabled.


In [3]:
# Load prepared arrays as memory maps so multi-gigabyte sequence files are not all copied into RAM.
city_dir = PREPARED_DIR / CITY
required = [city_dir / f"X_{split}.npy" for split in ("train", "validation", "test")]
required += [city_dir / f"y_{split}.npy" for split in ("train", "validation", "test")]
missing = [str(path) for path in required if not path.exists()]
assert not missing, "Missing prepared files:\n" + "\n".join(missing)

X_train = np.load(city_dir / "X_train.npy", mmap_mode="r")
X_val = np.load(city_dir / "X_validation.npy", mmap_mode="r")
X_test = np.load(city_dir / "X_test.npy", mmap_mode="r")
y_train = np.load(city_dir / "y_train.npy", mmap_mode="r")
y_val = np.load(city_dir / "y_validation.npy", mmap_mode="r")
y_test = np.load(city_dir / "y_test.npy", mmap_mode="r")
metadata = json.loads((city_dir / "metadata.json").read_text(encoding="utf-8"))
target_scaler = joblib.load(city_dir / "target_scaler.joblib")

for name, X, y in [("train", X_train, y_train), ("validation", X_val, y_val), ("test", X_test, y_test)]:
    assert X.ndim == 3 and y.ndim == 1
    assert len(X) == len(y) and len(X) > 0
    assert X.shape[1] == metadata["lookback_hours"]
    assert X.shape[2] == len(metadata["feature_columns"])
    print(f"{name:10s} X={X.shape}, y={y.shape}")

LOOKBACK, N_FEATURES = X_train.shape[1:]
print(f"City: {CITY}; lookback: {LOOKBACK}; features: {N_FEATURES}")

train      X=(48753, 168, 33), y=(48753,)
validation X=(10519, 168, 33), y=(10519,)
test       X=(10520, 168, 33), y=(10520,)
City: las_vegas; lookback: 168; features: 33


In [4]:
# Batch generator reads only the required pieces from each memory-mapped array.
def make_dataset(X, y, batch_size, training=False):
    indices = np.arange(len(y))
    def batch_generator():
        order = indices.copy()
        if training:
            np.random.default_rng(SEED).shuffle(order)
        for start in range(0, len(order), batch_size):
            batch_ids = order[start:start + batch_size]
            # Copy converts memmap slices to regular float32 arrays for TensorFlow.
            yield np.asarray(X[batch_ids], dtype=np.float32), np.asarray(y[batch_ids], dtype=np.float32)
    signature = (
        tf.TensorSpec(shape=(None, LOOKBACK, N_FEATURES), dtype=tf.float32),
        tf.TensorSpec(shape=(None,), dtype=tf.float32),
    )
    return tf.data.Dataset.from_generator(batch_generator, output_signature=signature).prefetch(tf.data.AUTOTUNE)

train_ds = make_dataset(X_train, y_train, BATCH_SIZE, training=True)
val_ds = make_dataset(X_val, y_val, BATCH_SIZE)
test_ds = make_dataset(X_test, y_test, BATCH_SIZE)

In [5]:
def tcn_residual_block(x, filters, dilation_rate, kernel_size=3, dropout=0.10):
    residual = x
    for _ in range(2):
        x = tf.keras.layers.Conv1D(filters, kernel_size, padding="causal", dilation_rate=dilation_rate, activation="relu")(x)
        x = tf.keras.layers.SpatialDropout1D(dropout)(x)
    if residual.shape[-1] != filters:
        residual = tf.keras.layers.Conv1D(filters, 1, padding="same")(residual)
    return tf.keras.layers.Activation("relu")(tf.keras.layers.Add()([residual, x]))

def build_tcn(lookback, n_features):
    inputs = tf.keras.Input(shape=(lookback, n_features), name="hourly_features")
    x = inputs
    for dilation in [1, 2, 4, 8, 16, 32, 64]:
        x = tcn_residual_block(x, filters=64, dilation_rate=dilation)
    x = tf.keras.layers.GlobalAveragePooling1D()(x)
    x = tf.keras.layers.Dense(64, activation="relu")(x)
    x = tf.keras.layers.Dropout(0.20)(x)
    # Keep the regression output in float32 when mixed precision is enabled.
    outputs = tf.keras.layers.Dense(1, dtype="float32", name="scaled_load")(x)
    return tf.keras.Model(inputs, outputs, name="electricity_demand_tcn")

# Build and compile explicitly on the GPU. With soft placement disabled, this fails rather than using CPU.
with tf.device("/GPU:0"):
    model = build_tcn(LOOKBACK, N_FEATURES)
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE), loss="mse", metrics=[tf.keras.metrics.MeanAbsoluteError(name="mae")])

# Verify the model can execute a forward pass on GPU before long training begins.
with tf.device("/GPU:0"):
    model_probe = model(tf.zeros((1, LOOKBACK, N_FEATURES), dtype=tf.float32), training=False)
assert "GPU" in model_probe.device.upper(), f"Model probe ran on {model_probe.device}, not GPU"
print(f"Model GPU execution verified: {model_probe.device}")
model.summary()

Model GPU execution verified: /job:localhost/replica:0/task:0/device:GPU:0
Model: "electricity_demand_tcn"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 hourly_features (InputLayer)   [(None, 168, 33)]    0           []                               
                                                                                                  
 conv1d (Conv1D)                (None, 168, 64)      6400        ['hourly_features[0][0]']        
                                                                                                  
 spatial_dropout1d (SpatialDrop  (None, 168, 64)     0           ['conv1d[0][0]']                 
 out1D)                                                                                           
                                                                                                  
 c

In [6]:
# Train. Validation monitoring selects the best epoch rather than the final epoch.
run_dir = TRAINING_DIR / CITY
run_dir.mkdir(parents=True, exist_ok=True)
checkpoint_path = run_dir / "best_tcn.keras"
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(checkpoint_path, monitor="val_loss", mode="min", save_best_only=True, verbose=1),
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", mode="min", patience=12, restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", mode="min", factor=0.5, patience=5, min_lr=1e-6, verbose=1),
    tf.keras.callbacks.CSVLogger(run_dir / "training_history.csv"),
]

# Do not wrap model.fit in tf.device(): tf.data bookkeeping has CPU-only operations.
# The model was built/compiled on GPU:0 and the GPU probe above verified its execution device.
history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=callbacks, verbose=1)
model.save(run_dir / "final_tcn.keras")
print(f"Best model and history saved in: {run_dir}")

Epoch 1/100
    762/Unknown - 29s 29ms/step - loss: 0.3606 - mae: 0.3854
Epoch 1: val_loss improved from inf to 0.07872, saving model to D:\Downloads\BTP\model_training\lstm_training_3\data\tcn_training\las_vegas\best_tcn.keras
762/762 [==============================] - 34s 35ms/step - loss: 0.3606 - mae: 0.3854 - val_loss: 0.0787 - val_mae: 0.2085 - lr: 0.0010
Epoch 2/100
762/762 [==============================] - ETA: 0s - loss: 0.0755 - mae: 0.2025
Epoch 2: val_loss improved from 0.07872 to 0.05306, saving model to D:\Downloads\BTP\model_training\lstm_training_3\data\tcn_training\las_vegas\best_tcn.keras
762/762 [==============================] - 26s 34ms/step - loss: 0.0755 - mae: 0.2025 - val_loss: 0.0531 - val_mae: 0.1744 - lr: 0.0010
Epoch 3/100
762/762 [==============================] - ETA: 0s - loss: 0.0527 - mae: 0.1685
Epoch 3: val_loss improved from 0.05306 to 0.04526, saving model to D:\Downloads\BTP\model_training\lstm_training_3\data\tcn_training\las_vegas\best_tcn.kera

In [7]:
# Final test evaluation in MW (not scaled units). The test set remains untouched during training.
best_model = tf.keras.models.load_model(run_dir / "best_tcn.keras")
# As with fit(), let tf.data run its required CPU bookkeeping while GPU-resident model layers execute on GPU.
scaled_predictions = best_model.predict(test_ds, verbose=1).reshape(-1)

actual_mw = target_scaler.inverse_transform(np.asarray(y_test).reshape(-1, 1)).ravel()
predicted_mw = target_scaler.inverse_transform(scaled_predictions.reshape(-1, 1)).ravel()
mae_mw = float(np.mean(np.abs(actual_mw - predicted_mw)))
rmse_mw = float(np.sqrt(np.mean((actual_mw - predicted_mw) ** 2)))
mape = float(np.mean(np.abs((actual_mw - predicted_mw) / actual_mw)) * 100)
ss_res = np.sum((actual_mw - predicted_mw) ** 2)
ss_tot = np.sum((actual_mw - np.mean(actual_mw)) ** 2)
r2 = float(1 - ss_res / ss_tot)
metrics = {"city": CITY, "MAE_MW": mae_mw, "RMSE_MW": rmse_mw, "MAPE_percent": mape, "R2": r2}
print(json.dumps(metrics, indent=2))
pd.DataFrame({"actual_mw": actual_mw, "predicted_mw": predicted_mw}).to_csv(run_dir / "test_predictions.csv", index=False)
(run_dir / "test_metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")

165/165 [==============================] - 2s 8ms/step
{
  "city": "las_vegas",
  "MAE_MW": 60.93434524536133,
  "RMSE_MW": 80.75519561767578,
  "MAPE_percent": 1.859649084508419,
  "R2": 0.9879555040970445
}


153

## Files produced

The city folder under `D:\Downloads\BTP\model_training\lstm_training_3\data\tcn_training` will contain `best_tcn.keras`, `final_tcn.keras`, `training_history.csv`, `test_predictions.csv`, and `test_metrics.json`. To train Phoenix, change `CITY` and restart from the first code cell.